# 04 — Evaluate downstream tasks

Once the model is trained, this notebook produces:

- per-task AUROC / AUPRC / F1 / ECE on the held-out test split
- calibration curves
- confusion matrices
- PCA embedding visualization
- comparison against classical baselines

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

### Load test results produced by `scripts/train_model.py`

In [ ]:
test_metrics = pd.read_csv('../results/tables/metrics_test.csv')
test_metrics

In [ ]:
baselines = pd.read_csv('../results/tables/baselines.csv')
baselines

### Side-by-side AUROC: foundation model vs baselines

In [ ]:
fm = test_metrics.set_index('task')[['auroc']].rename(columns={'auroc':'foundation_model'})
bl = baselines.pivot(index='task', columns='model', values='auroc')
comparison = fm.join(bl).sort_index()
ax = comparison.plot.bar(figsize=(8,4), rot=20)
ax.set_ylabel('AUROC'); ax.set_ylim(0.4, 1.0); ax.grid(axis='y', alpha=0.3)
ax.set_title('Per-task AUROC: foundation model vs classical baselines')
plt.tight_layout(); plt.show()
comparison.round(3)

### Calibration curves and confusion matrices

(Re-run the model on the test set to access raw probabilities for these plots.)

In [ ]:
import json
with open('../results/tables/metrics_test.json') as f:
    metrics_json = json.load(f)
list(metrics_json)

### Representation visualization (PCA)

Embed the test split's pooled representations and color by participant id
to verify the encoder is learning a coherent per-person structure.

In [ ]:
# In a notebook run, you'd load the checkpoint and call
# `extract_representations` from lhfm.training.evaluate. Here we just sketch
# the plotting code you'd use afterwards.
#
# from lhfm.training.evaluate import extract_representations
# reps, pids = extract_representations(model, test_ds, device='cpu')
# coords = PCA(n_components=2).fit_transform(reps)
# plt.scatter(coords[:,0], coords[:,1], c=pids, cmap='tab20', s=10, alpha=0.7)
# plt.title('Foundation-model representations (PCA-2)'); plt.show()
